# 10. W?asne middleware od potrzeby do dzia?aj?cego agenta

Zbudujemy jedn? warstw?, kt?ra ogranicza liczb? wywo?a? modelu w rozmowie. To ma?y, kompletny przyk?ad: wymaganie ? hook ? pod??czenie ? wynik ? sprawdzenie. Model jest skryptowany, wi?c lekcja nie wymaga klucza ani uruchomionego Ollamy. Ten model **nie podejmuje decyzji**: zwraca przygotowane odpowiedzi.

[Oficjalne hooki custom middleware](https://docs.langchain.com/oss/python/langchain/middleware/custom)

In [ ]:
# Wszystkie importy lekcji s? tutaj.
from typing import NotRequired

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver


## 1. Wybieramy punkt zaczepienia

Agent mo?e wo?a? model wiele razy. `before_model(state, runtime)` uruchamia si? przed ka?d? tak? pr?b?. Widzimy stan rozmowy i mo?emy zwr?ci? jego **cz??ciow? aktualizacj?**. Gdy bud?et si? sko?czy, skierujemy graf do ko?ca. `wrap_model_call(request, handler)` otacza faktyczne wywo?anie modelu; musi zwr?ci? wynik `handler(request)`, je?li nie zast?puje odpowiedzi ?wiadomie.

`before_agent` / `after_agent` dzia?aj? raz na wywo?anie agenta. `before_model` / `after_model` dzia?aj? wok?? ka?dej tury modelu. Wrappery modelu i narz?dzia dzia?aj? wok?? pojedynczych wywo?a?. Nie trzeba implementowa? wszystkich hook?w.

## 2. Projektujemy stan i kontrakt

`model_calls` nale?y do state, bo ma rosn?? w obr?bie rozmowy. `AgentState` ju? zawiera `messages`. Zwracamy tylko zmieniane pola; nie edytujemy s?ownika `state` w miejscu. Dzi?ki checkpointerowi licznik przejdzie do nast?pnego `invoke` z tym samym `thread_id`. Limit w konstruktorze jest konfiguracj? samego middleware.

In [ ]:
class BudgetState(AgentState):
    model_calls: NotRequired[int]


class ModelBudget(AgentMiddleware):
    state_schema = BudgetState

    def __init__(self, max_calls: int):
        if max_calls < 1:
            raise ValueError("max_calls must be positive")
        self.max_calls = max_calls
        self.events = []  # Tylko do obserwacji w tej lekcji, nie jest stanem rozmowy.

    @hook_config(can_jump_to=["end"])
    def before_model(self, state: BudgetState, runtime):
        used = state.get("model_calls", 0)
        if used >= self.max_calls:
            # Ten komunikat trafia do state, a skok ko?czy przebieg bez modelu.
            self.events.append("blocked")
            return {"messages": [AIMessage(content="Limit wywo?a? modelu osi?gni?ty.")], "jump_to": "end"}
        # Zwr?cony fragment stanu zostanie scalony przez graf przed wywo?aniem modelu.
        return {"model_calls": used + 1}

    def wrap_model_call(self, request, handler):
        self.events.append("model:enter")
        try:
            return handler(request)  # Jedno wywo?anie; nie powtarzamy go przypadkiem.
        finally:
            self.events.append("model:leave")


## 3. Pod??czamy do `create_agent`

Lista `middleware=[...]` m?wi agentowi, kiedy wywo?ywa? hooki. Checkpointer zachowuje `model_calls` razem z histori?. Skryptowany model ma dok?adnie dwie odpowiedzi. `events` pokazuje, czy wywo?anie modelu faktycznie nast?pi?o.

In [ ]:
budget = ModelBudget(max_calls=2)
model = FakeMessagesListChatModel(responses=[
    AIMessage(content="Pierwsza odpowied?."),
    AIMessage(content="Druga odpowied?."),
])
agent = create_agent(model=model, middleware=[budget], checkpointer=InMemorySaver())
thread = {"configurable": {"thread_id": "budget-demo"}}


In [ ]:
first = agent.invoke({"messages": [("user", "Pytanie 1")]}, config=thread)
first["model_calls"], first["messages"][-1].content, budget.events


In [ ]:
second = agent.invoke({"messages": [("user", "Pytanie 2")]}, config=thread)
second["model_calls"], second["messages"][-1].content, budget.events


In [ ]:
third = agent.invoke({"messages": [("user", "Pytanie 3")]}, config=thread)
third["model_calls"], third["messages"][-1].content, budget.events


Trzecie wywo?anie ko?czy si? w `before_model`; nie powinno doda? `model:enter`. Sprawd?my to bez zgadywania z samej odpowiedzi tekstowej.

In [ ]:
assert (first["model_calls"], second["model_calls"], third["model_calls"]) == (1, 2, 2)
assert budget.events == ["model:enter", "model:leave", "model:enter", "model:leave", "blocked"]
assert third["messages"][-1].content == "Limit wywo?a? modelu osi?gni?ty."


## 4. Nowa rozmowa, nowy bud?et

Instancja middleware jest wsp??dzielona, wi?c `self.events` gromadzi obserwacje **obu** rozm?w. Nie wolno u?ywa? takiego pola jako licznika uprawnie? lub limitu dla jednego u?ytkownika. W?a?ciwy licznik jest w `state`, oddzielonym przez `thread_id`.

In [ ]:
new_thread = {"configurable": {"thread_id": "budget-new"}}
new_result = agent.invoke({"messages": [("user", "Nowa rozmowa")]}, config=new_thread)
new_result["model_calls"], new_result["messages"][-1].content


## 5. Przepis na w?asne middleware

1. Nazwij jedno wymaganie i wybierz hook najbli?szy zdarzeniu (`before_model` do aktualizacji przed modelem, `wrap_tool_call` do kontroli konkretnego narz?dzia).
2. Ustal, kt?re dane nale?? do state, kt?re do `runtime.context`, a kt?re do konfiguracji warstwy. Dane per rozmowa nie powinny mieszka? w `self`.
3. Zdefiniuj minimalny schemat stanu tylko wtedy, gdy dodajesz pola. Hook w?z?owy zwraca cz??ciow? aktualizacj? lub `None`; wrapper zwraca odpowied? handlera albo jawny zamiennik.
4. Pod??cz middleware do `create_agent`, uruchom ma?y powtarzalny przyk?ad i sprawd? efekt oraz liczb? faktycznych wywo?a?.
5. Przy b??dach i ponowieniach pami?taj o efektach ubocznych oraz idempotencji. Przy `ainvoke` potrzebne s? odpowiednie hooki asynchroniczne.

Dalsze, trudniejsze przypadki (narz?dzia, kolejno?? wielu warstw, retry, async, streaming) s? w [warsztacie 16](06_middlewares/16_custom_middleware.ipynb).